## Data preparation

Load dataset

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

repo_id = "mks-logic/gender_prediction"
files = ["all.csv"]
repo_type = "dataset"


for f in files:
    path = hf_hub_download(repo_id=repo_id, repo_type=repo_type, filename=f)
    shutil.copy(path, f"./data/{f}")  # change dir here

Prepare data

In [3]:
import pandas as pd

df = pd.read_csv("data/all.csv")

In [4]:
from baseline import baseline_aggregates

X, y = baseline_aggregates(df)

In [6]:
from sklearn.model_selection import train_test_split

def split_data(X, y, random_state=42):
    """
    Splits the data into training, validation, and test sets.
    70% train, 10% val, 20% test
    """

    # First split: train + temp (temp will be split into val and test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=random_state
    )

    # Second split: val and test from the temp set
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=2./3, random_state=random_state
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = split_data(X, y)
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((5880, 272), (5880,), (840, 272), (840,), (1680, 272), (1680,))

## Train

In [7]:
from baseline import fit_predict_xgb

y_pred_proba, y_pred = fit_predict_xgb(X_train, y_train, X_val, y_val, X_test)

## Eval

In [8]:
from eval import eval

metrics = eval(y_test, y_pred, y_pred_proba)
metrics

{'auc_score': 0.8575158181446243,
 'accuracy_score': 0.7803571428571429,
 'avg_precision_score': 0.8460449009234063,
 'precision_score': 0.7834302325581395,
 'recall_score': 0.7101449275362319}

## Your solution

In [ ]:
def your_features(df):
    return X, y

def fit_predict_proba(X_train, y_train, X_val, y_val, X_test):
    # Fit the model

    # Predict probabilities
    y_pred_proba = ...
    y_pred = ...

    return y_pred_proba, y_pred

In [ ]:
# Feature engineering
X, y = your_features(df)

# Train/val/test split
X_train, y_train, X_val, y_val, X_test, y_test = split_data(X, y)

# Model training and prediction
y_pred_proba, y_pred = fit_predict_proba(X_train, y_train, X_val, y_val, X_test)

# Evaluation
metrics = eval(y_test, y_pred, y_pred_proba)
metrics